# Bouwen met de Meta Family-modellen

## Introductie

Deze les behandelt:

- Verkenning van de twee belangrijkste Meta Family-modellen - Llama 3.1 en Llama 3.2
- Begrip van de gebruikssituaties en scenario's voor elk model
- Codevoorbeeld om de unieke kenmerken van elk model te tonen


## De Meta Family van Modellen

In deze les verkennen we 2 modellen uit de Meta Family of "Llama Herd" - Llama 3.1 en Llama 3.2

Deze modellen zijn beschikbaar in verschillende varianten en te vinden in de [Microsoft Foundry Models catalogus](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Opmerking:** GitHub Models wordt uitgefaseerd eind juli 2026. Hier is meer informatie over het gebruik van [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) om te prototypen met AI-modellen.

Modelvarianten:
- Llama 3.1 - 70B Instruct
- Llama 3.1 - 405B Instruct
- Llama 3.2 - 11B Vision Instruct
- Llama 3.2 - 90B Vision Instruct

*Opmerking: Llama 3 is ook beschikbaar in Microsoft Foundry Models, maar wordt in deze les niet behandeld*



## Llama 3.1 

Met 405 miljard parameters valt Llama 3.1 in de categorie open source LLM. 

De modus is een upgrade van de eerdere release Llama 3 door het aanbieden van: 

- Groter contextvenster - 128k tokens versus 8k tokens 
- Groter maximaal aantal output tokens - 4096 versus 2048 
- Betere meertalige ondersteuning - door toename in trainings-tokens 

Hierdoor kan Llama 3.1 meer complexe gebruikssituaties aan bij het bouwen van GenAI-toepassingen, waaronder: 
- Native functie-aanroepen - de mogelijkheid om externe tools en functies buiten de LLM-werkstroom aan te roepen 
- Betere RAG-prestaties - door het grotere contextvenster 
- Synthese van gegevens - het vermogen om effectieve data te creëren voor taken zoals fine-tuning 



### Native Functieoproepen 

Llama 3.1 is verfijnd om effectiever te zijn bij het maken van functie- of toolaanroepen. Het heeft ook twee ingebouwde tools die het model kan herkennen als nodig op basis van de prompt van de gebruiker. Deze tools zijn: 

- **Brave Search** - Kan worden gebruikt om up-to-date informatie te verkrijgen, zoals het weer, door een webzoekopdracht uit te voeren 
- **Wolfram Alpha** - Kan worden gebruikt voor complexere wiskundige berekeningen, zodat het schrijven van je eigen functies niet nodig is. 

Je kunt ook je eigen aangepaste tools maken die de LLM kan aanroepen. 

In het onderstaande codevoorbeeld: 

- Definiëren we de beschikbare tools (brave_search, wolfram_alpha) in de systeem-prompt. 
- Verstuur een gebruikersprompt die vraagt naar het weer in een bepaalde stad. 
- De LLM zal reageren met een tool-oproep naar de Brave Search-tool die er zo uit zal zien `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Opmerking: dit voorbeeld doet alleen de tool-oproep, als je de resultaten wilt krijgen, moet je een gratis account aanmaken op de Brave API-pagina en de functie zelf definiëren` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Ondanks dat het een LLM is, heeft Llama 3.1 een beperking: multimodaliteit. Dat wil zeggen, het kunnen gebruiken van verschillende soorten input zoals afbeeldingen als prompts en het geven van reacties. Deze mogelijkheid is één van de belangrijkste kenmerken van Llama 3.2. Deze kenmerken omvatten ook: 

- Multimodaliteit - heeft de mogelijkheid om zowel tekst- als afbeeldingsprompts te evalueren 
- Kleine tot middelgrote variaties (11B en 90B) - dit biedt flexibele inzetmogelijkheden, 
- Alleen tekst variaties (1B en 3B) - dit maakt het mogelijk het model op edge-/mobiele apparaten te implementeren en zorgt voor lage latency 

De multimodale ondersteuning vertegenwoordigt een grote stap in de wereld van open source modellen. Het onderstaande codevoorbeeld neemt zowel een afbeelding als tekstprompt om een analyse van de afbeelding te verkrijgen van Llama 3.2 90B. 

### Multimodale ondersteuning met Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Het leren stopt hier niet, zet de reis voort

Na het voltooien van deze les, bekijk onze [Generative AI Learning collection](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) om je kennis van Generative AI verder te vergroten!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
